Importations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from re import sub
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from plot_partition import plot_partition

# Folder with the CSV files
data_dir = Path.cwd()

STEP 1: Plot congestion function

In [ ]:



def plot_congestion_snapshot(time_min_vec):
    """
    Plot congestion snapshots of the road network at specified times.

    Parameters
    ----------
    time_min_vec : list or array-like
        Requested snapshot times in minutes.

    Example
    -------
    plot_congestion_snapshot([60, 90, 120])

    Required files
    --------------
    links.csv
    nodes.csv
    occupancy.csv

    The CSV files should be located in the same folder as this
    Python file.

    Occupancy = 0%   -> white
    Occupancy = 100% -> black
    """

    # ----------------------------------------------------------
    # 1. Locate and read data
    # ----------------------------------------------------------



    # links.csv and nodes.csv contain numerical data without
    # a header row
    links = pd.read_csv(
        data_dir / "links.csv",
        header=None
    ).to_numpy()

    nodes = pd.read_csv(
        data_dir / "nodes.csv",
        header=None
    ).to_numpy()

    # occupancy.csv:
    # first row    = link IDs
    # first column = time [sec]
    occupancy_raw = pd.read_csv(
        data_dir / "occupancy.csv",
        header=None
    ).to_numpy()

    # ----------------------------------------------------------
    # 2. Network information
    # ----------------------------------------------------------

    # links.csv columns:
    # 0 = Link ID
    # 1 = Length
    # 2 = Number of lanes
    # 3 = Starting node ID
    # 4 = Ending node ID
    # 5 = Region

    link_id = links[:, 0].astype(int)
    start_node = links[:, 3].astype(int)
    end_node = links[:, 4].astype(int)

    # nodes.csv columns:
    # 0 = Node ID
    # 1 = x-coordinate
    # 2 = y-coordinate

    node_id = nodes[:, 0].astype(int)
    node_x = nodes[:, 1].astype(float)
    node_y = nodes[:, 2].astype(float)

    # ----------------------------------------------------------
    # 3. Occupancy data
    # ----------------------------------------------------------

    occupancy_link_id = occupancy_raw[0, 1:].astype(int)

    time_sec = occupancy_raw[1:, 0].astype(float)

    occupancy = occupancy_raw[1:, 1:].astype(float)

    # ----------------------------------------------------------
    # 4. Match occupancy columns with links.csv
    # ----------------------------------------------------------

    occupancy_id_to_col = {
        link: j
        for j, link in enumerate(occupancy_link_id)
    }

    occupancy_columns = []

    for link in link_id:

        if link not in occupancy_id_to_col:
            raise ValueError(
                f"Link {link} from links.csv was not found "
                f"in occupancy.csv."
            )

        occupancy_columns.append(
            occupancy_id_to_col[link]
        )

    # Reorder occupancy columns so they follow links.csv
    occupancy = occupancy[:, occupancy_columns]

    # ----------------------------------------------------------
    # 5. Map node IDs to coordinates
    # ----------------------------------------------------------

    node_coordinates = {
        node_id[i]: (node_x[i], node_y[i])
        for i in range(len(node_id))
    }

    segments = []

    for i in range(len(link_id)):

        s = start_node[i]
        e = end_node[i]

        if s not in node_coordinates:
            raise ValueError(
                f"Start node {s} was not found in nodes.csv."
            )

        if e not in node_coordinates:
            raise ValueError(
                f"End node {e} was not found in nodes.csv."
            )

        x1, y1 = node_coordinates[s]
        x2, y2 = node_coordinates[e]

        segments.append([
            [x1, y1],
            [x2, y2]
        ])

    # ----------------------------------------------------------
    # 6. Process requested times
    # ----------------------------------------------------------

    time_min_vec = np.atleast_1d(
        np.asarray(time_min_vec, dtype=float)
    )

    n_plots = len(time_min_vec)

    if n_plots == 0:
        raise ValueError(
            "time_min_vec must contain at least one time."
        )

    # ----------------------------------------------------------
    # 7. Determine subplot layout automatically
    # ----------------------------------------------------------

    if n_plots <= 3:

        n_rows = 1
        n_cols = n_plots

    else:

        n_rows = int(np.floor(np.sqrt(n_plots)))
        n_cols = int(np.ceil(n_plots / n_rows))

    # ----------------------------------------------------------
    # 8. Create figure
    # ----------------------------------------------------------

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(5 * n_cols, 5 * n_rows),
        squeeze=False
    )

    axes = axes.flatten()

    # Common occupancy-to-color mapping
    norm = Normalize(
        vmin=0,
        vmax=100
    )

    # Greys:
    # 0   -> white
    # 100 -> black
    cmap = plt.get_cmap("Greys")

    # ----------------------------------------------------------
    # 9. Plot requested snapshots
    # ----------------------------------------------------------

    for j, time_min in enumerate(time_min_vec):

        ax = axes[j]

        target_time_sec = time_min * 60

        # Check whether requested time is within simulation
        if (
            target_time_sec < np.min(time_sec)
            or
            target_time_sec > np.max(time_sec)
        ):

            ax.text(
                0.5,
                0.5,
                f"t = {time_min:g} min\n"
                f"is outside the simulation period",
                ha="center",
                va="center",
                transform=ax.transAxes
            )

            ax.axis("off")
            continue

        # Find nearest available measurement
        row = np.argmin(
            np.abs(time_sec - target_time_sec)
        )

        actual_time_sec = time_sec[row]
        actual_time_min = actual_time_sec / 60

        occupancy_vec = occupancy[row, :]

        # Restrict occupancy to [0, 100]
        occupancy_vec = np.clip(
            occupancy_vec,
            0,
            100
        )

        # ------------------------------------------------------
        # Plot road links
        # ------------------------------------------------------

        collection = LineCollection(
            segments,
            cmap=cmap,
            norm=norm,
            linewidths=1.5
        )

        # Occupancy directly controls link color
        collection.set_array(
            occupancy_vec
        )

        ax.add_collection(
            collection
        )

        ax.autoscale()
        ax.set_aspect("equal")
        ax.axis("off")

        # If requested time exactly matches measurement time
        if np.isclose(
            actual_time_sec,
            target_time_sec
        ):

            title_text = (
                f"t = {time_min:g} min"
            )

        # Otherwise show actual measurement time used
        else:

            title_text = (
                f"t = {actual_time_min:g} min "
                f"(nearest to {time_min:g} min)"
            )

        ax.set_title(
            title_text,
            fontweight="bold"
        )

    # ----------------------------------------------------------
    # 10. Hide unused subplot positions
    # ----------------------------------------------------------

    for j in range(n_plots, len(axes)):
        axes[j].axis("off")

    # ----------------------------------------------------------
    # 11. Shared colorbar
    # ----------------------------------------------------------

    sm = ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=axes.tolist(),
        fraction=0.025,
        pad=0.02
    )

    cbar.set_label(
        "Occupancy (%)"
    )

    cbar.set_ticks(
        [0, 20, 40, 60, 80, 100]
    )

    # ----------------------------------------------------------
    # 12. Figure title
    # ----------------------------------------------------------

    fig.suptitle(
        "Network Congestion",
        fontsize=14,
        fontweight="bold"
    )

    plt.show()








Use of function

In [ ]:
if __name__ == "__main__":
    plot_congestion_snapshot(
        [30, 60, 90, 120]
    )

STEP 2

In [ ]:
### Step 2
x=513
twolinks=[513, 516]
flow = pd.read_csv(
        data_dir / "flow.csv",
        header=None
    ).to_numpy()
occupancy = pd.read_csv(
        data_dir / "occupancy.csv",
        header=None
    ).to_numpy()
indexx= np.where(flow[0,:]==x)[0][0]
flowperhour = flow.copy()
flowperhour[1:,1:] = flow[1:,1:]*3600/90
plt.scatter(occupancy[1:,indexx],flowperhour[1:,indexx])
plt.title("Flow vs Occupancy for the link: "+str(flow[0][indexx]))
plt.xlabel("Occupancy [%]")
plt.ylabel("Flow [vehicles per hour]")
plt.show()

average_flow = np.mean(flowperhour[1:,1:], axis=1)
average_occupancy = np.mean(occupancy[1:,1:], axis=1)
links2 = pd.read_csv(
        data_dir / "links.csv",
        header=None
    )


indextwolinks= np.where(np.isin(flow[0,:],twolinks))[0]-1
averageflowtwolinks= np.mean(flowperhour[1:,indextwolinks], axis=1)
averageoccupancytwolinks= np.mean(occupancy[1:,indextwolinks], axis=1)
average_flow_by_region = {}
average_occupancy_by_region = {}

for region in sorted(links2[5].unique()):
    region_indices = links2.index[links2[5] == region].tolist()

    average_flow_by_region[region] = np.mean(
        flowperhour[1:, np.array(region_indices) + 1],
        axis=1
    )
    average_occupancy_by_region[region] = np.mean(
        occupancy[1:, np.array(region_indices) + 1],
        axis=1
    )


plt.scatter(averageoccupancytwolinks,averageflowtwolinks)
plt.title("Average Flow vs Average Occupancy for the links: "+str(twolinks))
plt.xlabel("Average Occupancy [%]")
plt.ylabel("Average Flow [vehicles per hour]")
plt.show()






plt.scatter(average_occupancy,average_flow)
plt.title("Average Flow vs Average Occupancy")
plt.xlabel("Average Occupancy [%]")
plt.ylabel("Average Flow [vehicles per hour]")
plt.show()
fig, axes = plt.subplots(
        2,
        2,
        figsize=(10,10),
        squeeze=False
    )
plt.title("Average Flow vs Average Occupancy by Region")
for i in range(1, 5):
    ax = axes[(i-1)//2, (i-1)%2]
    ax.scatter(average_occupancy_by_region[i], average_flow_by_region[i])
    ax.set_title("Region "+str(i))
    ax.set_xlabel("Average Occupancy [%]")
    ax.set_ylabel("Average Flow [vehicles per hour]")
    max_occupancy = 1.1*max(np.max(values) for values in average_occupancy_by_region.values())
    max_flow = 1.1*max(np.max(values) for values in average_flow_by_region.values())
    ax.set_xlim(0, max_occupancy)
    ax.set_ylim(0, max_flow)
plt.show()

STEP 3

In [ ]:


DETECTOR_LENGTH_M = 2.0
AVERAGE_VEHICLE_LENGTH_M = 5.0


def estimate_region_traffic(data_dir=data_dir, csv_path=None):
    links = pd.read_csv(
        data_dir / "links.csv",
        header=None,
        names=["link_id", "length_m", "lanes", "start_node", "end_node", "region"],
    )

    def read_measurements(filename):
        raw = pd.read_csv(data_dir / filename, header=None)
        link_ids = raw.iloc[0, 1:].astype(int).to_numpy()
        if len(np.unique(link_ids)) != len(link_ids):
            raise ValueError(f"Duplicate link IDs found in {filename}.")
        times_s = raw.iloc[1:, 0].astype(float).to_numpy()
        values = raw.iloc[1:, 1:].astype(float).to_numpy()
        return times_s, link_ids, values

    occupancy_times, occupancy_ids, occupancy = read_measurements("occupancy.csv")
    flow_times, flow_ids, flow = read_measurements("flow.csv")

    if not np.array_equal(occupancy_times, flow_times):
        raise ValueError("flow.csv and occupancy.csv have different time steps.")
    intervals_s = np.diff(flow_times)
    if len(intervals_s) == 0 or np.any(intervals_s <= 0):
        raise ValueError("Measurement times must contain positive intervals.")
    if not np.allclose(intervals_s, intervals_s[0]):
        raise ValueError("flow.csv must use a constant measurement interval.")

    link_ids = links["link_id"].astype(int).to_numpy()

    def align_columns(values, measurement_ids, filename):
        id_to_column = {link_id: index for index, link_id in enumerate(measurement_ids)}
        missing = [link_id for link_id in link_ids if link_id not in id_to_column]
        if missing:
            raise ValueError(f"Links missing from {filename}: {missing[:10]}")
        return values[:, [id_to_column[link_id] for link_id in link_ids]]

    occupancy = align_columns(occupancy, occupancy_ids, "occupancy.csv")
    flow = align_columns(flow, flow_ids, "flow.csv")
    flow_rate = flow * (3600.0 / intervals_s[0])

    lanes = links["lanes"].to_numpy(dtype=float)
    density = (
        (occupancy / 100.0)
        * lanes[None, :]
        / (AVERAGE_VEHICLE_LENGTH_M + DETECTOR_LENGTH_M)
        * 1000.0
    )
    dataframedensity = pd.DataFrame(density)
    dataframedensity.to_csv(data_dir / "density.csv", index=False, header=False)
    regions = links["region"].to_numpy()
    link_lengths = links["length_m"].to_numpy(dtype=float)
    time_min = occupancy_times / 60.0

    if csv_path is not None:
        n_times, n_links = density.shape
        link_density_table = pd.DataFrame(
            {
                "time_s": np.repeat(occupancy_times, n_links),
                "time_min": np.repeat(time_min, n_links),
                "link_id": np.tile(link_ids, n_times),
                "region": np.tile(regions, n_times),
                "link_length_m": np.tile(link_lengths, n_times),
                "lanes": np.tile(lanes, n_times),
                "occupancy_percent": occupancy.reshape(-1),
                "average_vehicle_length_m": AVERAGE_VEHICLE_LENGTH_M,
                "detector_length_m": DETECTOR_LENGTH_M,
                "density_veh_per_km": density.reshape(-1),
            }
        )
        link_density_table.to_csv(
            csv_path,
            index=False,
        )

    records = []

    for region in np.sort(pd.unique(regions)):
        in_region = regions == region
        lengths = link_lengths[in_region]
        density_length = density[:, in_region] * lengths[None, :]
        denominator = density_length.sum(axis=1)
        mean_density = denominator / lengths.sum()
        mean_speed = np.divide(
            (flow_rate[:, in_region] * lengths[None, :]).sum(axis=1),
            denominator,
            out=np.full(len(time_min), np.nan),
            where=denominator > 0,
        )

        records.extend(
            {
                "time_min": time,
                "region": region,
                "mean_density_veh_per_km": average_density,
                "mean_speed_km_per_h": average_speed,
            }
            for time, average_density, average_speed in zip(
                time_min, mean_density, mean_speed
            )
        )

    return pd.DataFrame(records)


def plot_region_traffic(results):
    regions = np.sort(results["region"].unique())
    n_regions = len(regions)
    n_cols = int(np.ceil(np.sqrt(n_regions)))
    n_rows = int(np.ceil(n_regions / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 4.5 * n_rows),
        squeeze=False,
    )
    axes = axes.ravel()
    time_min = results["time_min"]
    scatter = None

    for ax, region in zip(axes, regions):
        region_data = results[results["region"] == region]
        scatter = ax.scatter(
            region_data["mean_density_veh_per_km"],
            region_data["mean_speed_km_per_h"],
            c=region_data["time_min"],
            cmap="viridis",
            vmin=time_min.min(),
            vmax=time_min.max(),
            s=22,
            alpha=0.8,
        )
        ax.set_title(f"Region {region}")
        ax.set_xlabel("Mean density [veh/km]")
        ax.set_ylabel("Mean speed [km/h]")
        ax.grid(True, alpha=0.25)

    for ax in axes[n_regions:]:
        ax.set_visible(False)

    fig.suptitle("Regional mean speed vs. mean density")
    fig.subplots_adjust(top=0.88, right=0.87, wspace=0.3, hspace=0.35)
    colorbar_ax = fig.add_axes([0.90, 0.2, 0.025, 0.65])
    fig.colorbar(scatter, cax=colorbar_ax, label="Time [min]")

    fig, ax = plt.subplots(figsize=(11, 5.5))
    for region in regions:
        region_data = results[results["region"] == region]
        ax.plot(
            region_data["time_min"],
            region_data["mean_speed_km_per_h"],
            label=f"Region {region}",
        )

    ax.set_title("Regional mean speed over time")
    ax.set_xlabel("Time [min]")
    ax.set_ylabel("Mean speed [km/h]")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Region", ncol=2)
    fig.tight_layout()
    plt.show()


if __name__ == "__main__":
    csv_path = data_dir / "link_density_estimates.csv"
    regional_results = estimate_region_traffic(csv_path=csv_path)
    valid_speeds = regional_results["mean_speed_km_per_h"].dropna()
    print(f"Link density table saved to: {csv_path}")
    print(
        "Regional mean speed range: "
        f"{valid_speeds.min():.2f} to {valid_speeds.max():.2f} km/h"
    )
    plot_region_traffic(regional_results)

STEP 4

In [ ]:
def load_traffic_data():
    
   

    links = pd.read_csv(
        data_dir / "links.csv",
        header=None,
        names=["link_id", "length_m", "lanes", "start_node", "end_node", "region"],
    )

    flow_raw = pd.read_csv(data_dir / "flow.csv", header=None)
    occupancy_raw = pd.read_csv(data_dir / "occupancy.csv", header=None)

    flow_ids = flow_raw.iloc[0, 1:].astype(int).to_numpy()
    occupancy_ids = occupancy_raw.iloc[0, 1:].astype(int).to_numpy()

    flow_times = flow_raw.iloc[1:, 0].to_numpy(dtype=float)
    occupancy_times = occupancy_raw.iloc[1:, 0].to_numpy(dtype=float)

    flow_matrix = flow_raw.iloc[1:, 1:].to_numpy(dtype=float)
    occupancy_matrix = occupancy_raw.iloc[1:, 1:].to_numpy(dtype=float)

    links_order = links["link_id"].to_numpy(dtype=int)

    flow_lookup = {link_id: idx for idx, link_id in enumerate(flow_ids)}
    occupancy_lookup = {link_id: idx for idx, link_id in enumerate(occupancy_ids)}

    flow_aligned = np.zeros((len(flow_times), len(links_order)))
    occupancy_aligned = np.zeros((len(occupancy_times), len(links_order)))

    for j, link_id in enumerate(links_order):
        if link_id not in flow_lookup:
            raise ValueError(f"Link {link_id} is missing from flow.csv")
        if link_id not in occupancy_lookup:
            raise ValueError(f"Link {link_id} is missing from occupancy.csv")

        flow_aligned[:, j] = flow_matrix[:, flow_lookup[link_id]]
        occupancy_aligned[:, j] = occupancy_matrix[:, occupancy_lookup[link_id]]

    return links, flow_times, occupancy_times, flow_aligned, occupancy_aligned


def compute_link_metrics(links, flow_matrix, occupancy_matrix):
    """Compute accumulation and production per link across time."""
    length_m = links["length_m"].to_numpy(dtype=float)
    lanes = links["lanes"].to_numpy(dtype=float)
    density_veh_per_km = (occupancy_matrix / 100.0 * lanes[np.newaxis, :]) / 7.0 * 1000.0

    accumulation = density_veh_per_km * length_m[np.newaxis, :] / 1000.0
    production = flow_matrix * (3600.0 / 90.0) * length_m[np.newaxis, :] / 1000.0

    return accumulation, production


def plot_production_accumulation():

    links, _, _, flow_matrix, occupancy_matrix = load_traffic_data()
    accumulation, production = compute_link_metrics(links, flow_matrix, occupancy_matrix)

    rng = np.random.default_rng(42)
    all_link_ids = links["link_id"].to_numpy(dtype=int)

    random_link_id = int(rng.choice(all_link_ids))
    random_link_index = int(np.where(all_link_ids == random_link_id)[0][0])

    eligible_links = np.delete(all_link_ids, np.where(all_link_ids == random_link_id)[0][0])
    second_link_id = int(rng.choice(eligible_links))
    second_link_index = int(np.where(all_link_ids == second_link_id)[0][0])

    fig = plt.figure(figsize=(18, 12), constrained_layout=True)

    regions = sorted(links["region"].unique())
    n_regions = len(regions)
    ncols = min(3, n_regions)
    nrows = int(np.ceil(n_regions / ncols)) + 1
    gs = fig.add_gridspec(nrows, 3)

    ax = fig.add_subplot(gs[0, 0])
    ax.scatter(
        accumulation[:, random_link_index],
        production[:, random_link_index],
        s=18,
        alpha=0.7,
        color="tab:blue",
    )
    ax.set_xlabel("Accumulation (veh)")
    ax.set_ylabel("Production (veh·km/h)")
    ax.set_title(f"Single link {random_link_id}")
    ax.grid(True, alpha=0.2)

    ax = fig.add_subplot(gs[0, 1])
    acc_sum = accumulation[:, random_link_index] + accumulation[:, second_link_index]
    prod_sum = production[:, random_link_index] + production[:, second_link_index]
    ax.scatter(acc_sum, prod_sum, s=18, alpha=0.7, color="tab:orange")
    ax.set_xlabel("Accumulation (veh)")
    ax.set_ylabel("Production (veh·km/h)")
    ax.set_title(f"Sum of links {random_link_id} and {second_link_id}")
    ax.grid(True, alpha=0.2)

    for i, region in enumerate(regions):
        row = i // ncols
        col = i % ncols
        ax_region = fig.add_subplot(gs[1 + row, col])

        region_link_ids = links.loc[links["region"] == region, "link_id"].to_numpy(dtype=int)
        region_indices = [int(np.where(all_link_ids == link_id)[0][0]) for link_id in region_link_ids]

        region_accumulation = accumulation[:, region_indices].sum(axis=1)
        region_production = production[:, region_indices].sum(axis=1)

        ax_region.scatter(region_accumulation, region_production, s=16, alpha=0.7, color="tab:green")
        ax_region.set_title(f"Region {region}")
        ax_region.set_xlabel("Accumulation (veh)")
        ax_region.set_ylabel("Production (veh·km/h)")
        ax_region.grid(True, alpha=0.2)

    fig.suptitle("Production vs. accumulation", y=1.02, fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.savefig(data_dir / "production_vs_accumulation.png", dpi=200)
    plt.show()


if __name__ == "__main__":
    plot_production_accumulation()


STEP 5

In [ ]:
SAMPLE_FRACTION = 0.5
STRATEGIES = ["Highest lanes", "Longest length", "Max avg flow"]


def select_links(links, flow_matrix, region_idx, strategy):
    """Indices (into the full link arrays) of the sampled links of one region."""
    n_select = int(np.ceil(SAMPLE_FRACTION * len(region_idx)))
    lanes = links["lanes"].to_numpy(dtype=float)[region_idx]
    length = links["length_m"].to_numpy(dtype=float)[region_idx]
    avg_flow = flow_matrix[:, region_idx].mean(axis=0)

    if strategy == "Highest lanes":
        keys = (length, lanes)  # ties on lanes broken by longer length
    elif strategy == "Longest length":
        keys = (lanes, length)
    else:
        keys = (length, avg_flow)
    order = np.lexsort(keys)[::-1]
    return region_idx[order[:n_select]]


def error_metrics(true, est):
    err = est - true
    nonzero = true != 0
    pct = 100 * err[nonzero] / true[nonzero]
    return {
        "MAE": np.mean(np.abs(err)),
        "RMSE": np.sqrt(np.mean(err ** 2)),
        "MAPE [%]": np.mean(np.abs(pct)),
        "Percent error [%]": np.mean(pct),
    }


def estimate_partial_mfd():
    links, _, _, flow_matrix, occupancy_matrix = load_traffic_data()
    accumulation, production = compute_link_metrics(links, flow_matrix, occupancy_matrix)
    lengths = links["length_m"].to_numpy(dtype=float)
    regions = sorted(links["region"].unique())

    fig, axes = plt.subplots(
        len(regions), len(STRATEGIES), figsize=(5 * len(STRATEGIES), 4 * len(regions)),
        squeeze=False, constrained_layout=True,
    )
    rows = []

    for i, region in enumerate(regions):
        region_idx = np.where(links["region"].to_numpy() == region)[0]
        true_acc = accumulation[:, region_idx].sum(axis=1)
        true_prod = production[:, region_idx].sum(axis=1)

        for j, strategy in enumerate(STRATEGIES):
            sel = select_links(links, flow_matrix, region_idx, strategy)
            scale = lengths[region_idx].sum() / lengths[sel].sum()
            est_acc = scale * accumulation[:, sel].sum(axis=1)
            est_prod = scale * production[:, sel].sum(axis=1)

            ax = axes[i, j]
            ax.scatter(true_acc, true_prod, s=14, alpha=0.5, color="lightgray", label="Step 4 (all links)")
            ax.scatter(est_acc, est_prod, s=14, alpha=0.7, color="tab:red", label="Estimated")
            ax.set_title(f"Region {region} - {strategy} (scale {scale:.2f})")
            ax.set_xlabel("Accumulation (veh)")
            ax.set_ylabel("Production (veh·km/h)")
            ax.grid(True, alpha=0.2)
            if i == 0 and j == 0:
                ax.legend()

            for variable, t, e in (("Accumulation", true_acc, est_acc), ("Production", true_prod, est_prod)):
                rows.append({"Region": region, "Strategy": strategy, "Variable": variable,
                             **error_metrics(t, e)})

    fig.suptitle("MFD estimation from partial measurements", fontsize=14, fontweight="bold")
    plt.show()
    return pd.DataFrame(rows)


if __name__ == "__main__":
    step5_errors = estimate_partial_mfd()
    display(step5_errors.round(2).set_index(["Region", "Strategy", "Variable"]))


STEP 6

In [ ]:


flow = pd.read_csv(
        data_dir / "flow.csv",
        header=None
    ).to_numpy()
occupancy = pd.read_csv(
        data_dir / "occupancy.csv",
        header=None
    ).to_numpy()
links = pd.read_csv(
        data_dir / "links.csv",
        header=None
    ).to_numpy()
nodes = pd.read_csv(
        data_dir / "nodes.csv",
        header=None
    ).to_numpy()
density = pd.read_csv(
        data_dir / "density.csv",
        header=None
    ).to_numpy()

np.mean(occupancy[1:,1:], axis=0)
np.std(occupancy[1:,1:], axis=0)
meandensity=np.mean(density[:,:], axis=0)


standardized_mean_density = np.zeros_like(meandensity)

standardized_mean_density[:] = (meandensity - np.mean(meandensity[:], axis=0)) / np.std(meandensity, axis=0)

matrice = standardized_mean_density[:].astype(float)
matrice = np.nan_to_num(matrice, nan=0.0, posinf=0.0, neginf=0.0)
links2 = pd.read_csv(
        data_dir / "links.csv",
        header=None
    )
nodes2 = pd.read_csv(
        data_dir / "nodes.csv",
        header=None
    )
df_left = pd.merge(links2, nodes2, left_on=3, right_on=0, how='left')
df_final = pd.merge(df_left, nodes2, left_on=4, right_on=0, how='left')
df_final['meandensity'] = meandensity
numberk=[2,3,4,5,6]
wcss=[]
for k in numberk:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=1).fit(matrice.reshape(-1, 1))
    df_final[f'cluster_{k}'] = kmeans.labels_
    wcss.append(kmeans.inertia_)

plt.plot(numberk, wcss, marker="o")
plt.title("Elbow curve")
plt.xlabel("Number of clusters K")
plt.ylabel("WCSS")
plt.show()

# links sharing a node are neighbours (direction ignored)
node_index = {n: i for i, n in enumerate(nodes2[0])}
incidence = csr_matrix(
    (np.ones(2 * len(links2)), (np.r_[links2.index, links2.index], np.r_[links2[3].map(node_index), links2[4].map(node_index)])),
    shape=(len(links2), len(nodes2)),
)
adjacency = (incidence @ incidence.T).tocsr()

def n_pieces(labels):
    labels = np.asarray(labels)
    return [connected_components(adjacency[labels == c][:, labels == c], directed=False)[0] for c in np.unique(labels)]









cluster_colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]

for k in numberk:
    labels = df_final[f"cluster_{k}"]

    print(f"K={k}: connected pieces per cluster {n_pieces(labels)}")
    plot_partition(labels)



    

    rows = int(np.ceil(k / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(10, 3* rows), squeeze=False)
    for i in range(k):
        ax = axes[i // 2, i % 2]
        ax.set_title(f"Cluster {i}")
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_aspect("equal", adjustable="box")

        for _, sub in df_final.iterrows():
            if sub[f"cluster_{k}"] == i:
                xs = [sub.iloc[7], sub.iloc[10]]
                ys = [sub.iloc[8], sub.iloc[11]]
                ax.plot(
                    xs,
                    ys,
                    color=cluster_colors[i % len(cluster_colors)],
                    linewidth=3,
                    alpha=1,
                    zorder=1,
                )
        ax.scatter(nodes2[1], nodes2[2], s=1, zorder=2, color="black")

    for i in range(k, rows * 2):
        axes[i // 2, i % 2].set_visible(False)
    plt.tight_layout()
    plt.show()


STEP 7

In [ ]:

df_final['xmean']=(df_final.iloc[:, 10] + df_final.iloc[:, 7])/2
df_final['ymean']=(df_final.iloc[:, 11] + df_final.iloc[:, 8])/2
df_final['xmeannormalized']=(df_final['xmean']-np.mean(df_final['xmean']))/np.std(df_final['xmean'])
df_final['ymeannormalized']=(df_final['ymean']-np.mean(df_final['ymean']))/np.std(df_final['ymean'])
df_final["standardized_meandensity"] = (df_final["meandensity"] - np.mean(df_final["meandensity"])) / np.std(df_final["meandensity"])

def all_connected(w, k):
    matrice = df_final[['xmeannormalized', 'ymeannormalized', 'standardized_meandensity']].to_numpy()
    matrice[:, 0]=w*matrice[:, 0]
    matrice[:, 1]=w*matrice[:, 1]
    labels = KMeans(n_clusters=k, n_init=10, random_state=1).fit_predict(matrice)
    return all(p == 1 for p in n_pieces(labels))

# smallest w for which all k clusters are connected, steps of 0.1, then refine with 0.01 steps
kpoint=[]
for k in [2,3,4,5,6]:
    w = next(w for w in np.round(np.arange(0, 8.05, 0.1), 2) if all_connected(w, k))
    w = next(v for v in np.round(np.arange(max(w - 0.1, 0), w + 0.005, 0.01), 2) if all_connected(v, k))
    kpoint.append(w)
    

numberk=[2,3,4,5,6]
for k in numberk:
    matrice = df_final[['xmeannormalized', 'ymeannormalized', 'standardized_meandensity']].to_numpy()
    
    matrice[:, 0]=kpoint[k-2]*matrice[:, 0]
    matrice[:, 1]=kpoint[k-2]*matrice[:, 1]
    
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=1).fit_predict(matrice)
    
    df_final[f'clusterxy_{k}'] = kmeans


segments = []
for i in range(len(df_final)):
    x1 = df_final.iloc[i, 5]
    y1 = df_final.iloc[i, 6]
    x2 = df_final.iloc[i, 8]
    y2 = df_final.iloc[i, 9]
    segments.append([(x1, y1), (x2, y2)])


plt.figure(figsize=(8, 6))
for i, sub in df_final.iterrows():
    xs = [sub.iloc[7], sub.iloc[10]]
    ys = [sub.iloc[8], sub.iloc[11]]
    cluster = int(sub[f"clusterxy_{k}"])
    plt.plot(
        xs,
        ys,
        color=plt.cm.viridis(
            plt.Normalize(np.nanmin(meandensity), np.nanmax(meandensity))(meandensity[i])
        ),
        linewidth=3,
        alpha=1,
        zorder=1,
    )
plt.show()

plt.scatter(nodes2[1], nodes2[2], s=1, zorder=2, color="black")
plt.title(f"Clustered road links for k={k} with w_conn={kpoint[k-2]}")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()




cluster_colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]

for k in numberk:
    labels = df_final[f"clusterxy_{k}"]

    print(f"k={k}, w_conn={kpoint[k-2]}: connected pieces per cluster {n_pieces(labels)}")
    plot_partition(labels)


# MFD (as in step 4) of the k=4 connected partition vs the original 4 regions
links_mfd, _, _, flow_matrix, occupancy_matrix = load_traffic_data()
accumulation, production = compute_link_metrics(links_mfd, flow_matrix, occupancy_matrix)
time_min = (np.arange(accumulation.shape[0]) + 1) * 1.5  # 90-s measurement steps
partitions = [
    ("(a) Original regions (N=4)", "Region", links_mfd["region"]),
    ("(b) K-means clusters with spatial weight (k=4)", "Cluster", df_final["clusterxy_4"]),
]

# Two separate groups (columns)
fig = plt.figure(figsize=(12, 16), layout="constrained")
main, cbar_fig = fig.subfigures(1, 2, width_ratios=[12, 1])
groups = main.subfigures(1, 2, wspace=0.04)
scatter = None
for group, (title, label, partition) in zip(groups, partitions):
    group.suptitle(title, fontweight="bold")
    axes = group.subplots(4, 1)
    for ax, c in zip(axes, np.unique(partition)):
        mask = (partition == c).to_numpy()
        acc = accumulation[:, mask].sum(axis=1)
        prod = production[:, mask].sum(axis=1)
        coeffs = np.polyfit(acc, prod, 2)
        residual = prod - np.polyval(coeffs, acc)
        r2 = 1 - residual.var() / prod.var()
        cv = np.nanmean(density[:, mask].std(axis=1) / density[:, mask].mean(axis=1))
        print(f"{label} {c}: R2 of quadratic fit={r2:.3f}, mean density CV={cv:.2f}")

        scatter = ax.scatter(acc, prod, s=14, c=time_min, cmap="viridis", vmin=time_min.min(), vmax=time_min.max(), zorder=2)
        ax.text(0.04, 0.96, f"$R^2$ = {r2:.2f}\nCV = {cv:.2f}", transform=ax.transAxes, va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
        ax.set_title(f"{label} {c}")
        ax.set_xlabel("Accumulation (veh)")
        ax.set_ylabel("Production (veh·km/h)")
        ax.grid(True, alpha=0.2)

cbar_fig.colorbar(scatter, cax=cbar_fig.add_axes([0.05, 0.1, 0.25, 0.8]), label="Time [min]")
main.suptitle(r"Production vs. accumulation: original regions vs. clusters", fontsize=14, fontweight="bold")
plt.show()
